In [1]:
from scipy.stats import entropy
import pandas as pd
import numpy as np
import joblib

import config
from src.data import pull_statcast
from src.features import build_features, get_feature_cols
from src.arsenal import compute_arsenal

In [2]:
def pitcher_predictability(model, df, feature_cols):
    # get predicted probabilities for every pitch
    probs = model.predict_proba(df[feature_cols])
    
    # compute entropy per pitch (higher = less predictable)
    df = df.copy()
    df["entropy"] = [entropy(p) for p in probs]
    
    # average entropy per pitcher
    pitcher_entropy = (
        df.groupby("pitcher")["entropy"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "avg_entropy", "count": "n_pitches"})
    )

    pitcher_entropy["pitcher_name"] = pitcher_entropy.index.map(
        df.drop_duplicates("pitcher").set_index("pitcher")["player_name"]
    )

    # filter to pitchers with enough pitches to be meaningful
    pitcher_entropy = pitcher_entropy[
        pitcher_entropy["n_pitches"] >= config.MIN_TOTAL_PITCHES
    ]
    
    pitcher_entropy = pitcher_entropy.sort_values("avg_entropy", ascending=False)
    
    return pitcher_entropy

In [3]:
df = pull_statcast(config.SEASON, override_cache_path="../data/statcast_2025.parquet")
arsenal = compute_arsenal(df)
df = build_features(df, arsenal)
feature_cols = get_feature_cols(df, arsenal)

model = joblib.load("../models/xgb_pitch_predictor.pkl")

Loading cached data from ../data/statcast_2025.parquet


In [4]:
pitcher_entropy = pitcher_predictability(model, df, feature_cols)

In [9]:

print("Most unpredictable:")
print(pitcher_entropy.head(10)[["pitcher_name", "avg_entropy", "n_pitches"]])

print("\nMost predictable:")
print(pitcher_entropy.tail(10)[["pitcher_name", "avg_entropy", "n_pitches"]][::-1])

Most unpredictable:
              pitcher_name  avg_entropy  n_pitches
pitcher                                           
506433         Darvish, Yu     1.840757       1177
607625          Lugo, Seth     1.788106       2464
547179   Lorenzen, Michael     1.762589       2396
621111     Buehler, Walker     1.705697       2254
608331          Fried, Max     1.677789       3408
607259      Martinez, Nick     1.658707       2670
621107         Eflin, Zach     1.647725       1172
543243         Gray, Sonny     1.647478       2798
681190      Vásquez, Randy     1.647125       2148
641672       Hatch, Thomas     1.627771        586

Most predictable:
              pitcher_name  avg_entropy  n_pitches
pitcher                                           
592454       Kahnle, Tommy     0.436339       1093
656629     Kopech, Michael     0.469846        231
663362       Waldron, Matt     0.539645        104
519141      Pomeranz, Drew     0.543355        918
657612           Hill, Tim     0.578346    